# 03 — Monte Carlo Simulation

**Phase 3 (Weeks 3–4)** of the Portuguese wildfire catastrophe loss model.

Goals:
- Sample annual frequency → per-fire severities → aggregate annual loss
- Run 10,000+ scenarios
- Compute VaR(90), VaR(95), VaR(99), Expected Shortfall(95), skewness, kurtosis, max loss
- Plot loss distribution, Q-Q plots, tail comparisons

Success criteria (per PRD): Monte Carlo output converges — standard error on
VaR(95%) < 2% with 10k runs.

## Imports

In [ ]:
import json
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

from pathlib import Path

MODELS_DIR = Path("../models")
SIMULATION_DIR = Path("../simulation")

N_SCENARIOS = 10_000
RNG = np.random.default_rng(seed=42)

## Load fitted model parameters

In [ ]:
def load_model_params(filename: str) -> dict:
    """Load fitted distribution parameters saved in 02_distribution_fitting.ipynb.

    Parameters
    ----------
    filename : str
        JSON filename under MODELS_DIR (e.g. "poisson_frequency.json").

    Returns
    -------
    dict
        Parameter dictionary as saved by save_model_params.
    """
    # TODO: depends on models/*.json from Phase 2
    raise NotImplementedError("Fitted model parameters not yet available; complete 02_distribution_fitting.ipynb first")

## Simulation engine

In [ ]:
def simulate_annual_frequency(lam: float, n_scenarios: int, rng: np.random.Generator = RNG) -> np.ndarray:
    """Sample the number of fire events per scenario-year from a Poisson process.

    Parameters
    ----------
    lam : float
        Poisson rate parameter (fitted annual occurrence rate).
    n_scenarios : int
        Number of scenario-years to simulate.
    rng : np.random.Generator
        Random generator for reproducibility.

    Returns
    -------
    np.ndarray
        Shape (n_scenarios,) array of simulated fire counts.
    """
    # TODO: implement with rng.poisson(lam, size=n_scenarios)
    raise NotImplementedError("Frequency simulation pending fitted Poisson lambda")

In [ ]:
def simulate_severities(n_fires: int, severity_params: dict, rng: np.random.Generator = RNG) -> np.ndarray:
    """Sample per-fire loss severities from the fitted Lognormal (with Pareto tail).

    Parameters
    ----------
    n_fires : int
        Number of fire events to sample severities for.
    severity_params : dict
        Fitted Lognormal (and optionally Pareto tail) parameters.
    rng : np.random.Generator
        Random generator for reproducibility.

    Returns
    -------
    np.ndarray
        Shape (n_fires,) array of simulated per-event losses (EUR).
    """
    # TODO: implement Lognormal body + Pareto tail blending above threshold
    raise NotImplementedError("Severity simulation pending fitted severity parameters")

In [ ]:
def run_monte_carlo(frequency_params: dict, severity_params: dict, n_scenarios: int = N_SCENARIOS,
                     rng: np.random.Generator = RNG) -> np.ndarray:
    """Run the full frequency-severity Monte Carlo simulation.

    For each scenario-year: sample fire count (frequency model), sample a
    severity per fire (severity model), and sum to an aggregate annual loss.

    Parameters
    ----------
    frequency_params : dict
        Fitted Poisson parameters.
    severity_params : dict
        Fitted Lognormal/Pareto severity parameters.
    n_scenarios : int
        Number of scenario-years to simulate.
    rng : np.random.Generator
        Random generator for reproducibility.

    Returns
    -------
    np.ndarray
        Shape (n_scenarios,) array of simulated aggregate annual losses (EUR).
    """
    # TODO: loop scenarios calling simulate_annual_frequency + simulate_severities, sum per scenario
    raise NotImplementedError("Monte Carlo engine pending frequency and severity simulators")

## Risk metrics

In [ ]:
def compute_risk_metrics(annual_losses: np.ndarray) -> dict:
    """Compute portfolio-level risk metrics from simulated annual losses.

    Parameters
    ----------
    annual_losses : np.ndarray
        Simulated aggregate annual losses, one per scenario.

    Returns
    -------
    dict
        {"VaR_90": float, "VaR_95": float, "VaR_99": float,
         "ES_95": float, "skewness": float, "kurtosis": float,
         "max_loss": float}
    """
    # TODO: VaR via np.percentile; ES_95 as mean of losses above VaR_95;
    # skewness/kurtosis via scipy.stats.skew / scipy.stats.kurtosis
    raise NotImplementedError("Risk metrics pending simulated loss array")

## Plots

In [ ]:
def plot_loss_distribution(annual_losses: np.ndarray, metrics: dict = None, ax=None):
    """Plot a histogram of simulated annual losses, optionally marking VaR/ES lines.

    Parameters
    ----------
    annual_losses : np.ndarray
        Simulated aggregate annual losses.
    metrics : dict, optional
        Output of compute_risk_metrics; if given, VaR(95%) and ES(95%) are
        drawn as vertical reference lines.
    ax : matplotlib.axes.Axes, optional
        Axes to plot on; a new figure/axes is created if omitted.
    """
    if ax is None:
        _, ax = plt.subplots()
    ax.hist(annual_losses, bins=100)
    if metrics is not None:
        ax.axvline(metrics["VaR_95"], linestyle="--", label="VaR(95%)")
        ax.axvline(metrics["ES_95"], linestyle=":", label="ES(95%)")
        ax.legend()
    ax.set_xlabel("Aggregate annual loss (EUR)")
    ax.set_ylabel("Scenario count")
    ax.set_title("Simulated portfolio loss distribution")
    return ax

In [ ]:
def plot_qq(annual_losses: np.ndarray, dist=stats.lognorm, dist_params: tuple = None, ax=None):
    """Plot a quantile-quantile plot of simulated losses against a reference distribution.

    Parameters
    ----------
    annual_losses : np.ndarray
        Simulated aggregate annual losses.
    dist : scipy.stats rv_continuous
        Reference distribution for comparison.
    dist_params : tuple, optional
        Parameters for `dist`; required unless dist supports parameterless use.
    ax : matplotlib.axes.Axes, optional
        Axes to plot on; a new figure/axes is created if omitted.
    """
    if ax is None:
        _, ax = plt.subplots()
    # TODO: implement via scipy.stats.probplot(annual_losses, dist=dist, sparams=dist_params, plot=ax)
    raise NotImplementedError("Q-Q plot pending simulated loss array and reference distribution")

In [ ]:
def plot_tail_comparison(annual_losses: np.ndarray, historical_losses: np.ndarray, ax=None):
    """Compare simulated vs. historical loss distributions in the upper tail.

    Parameters
    ----------
    annual_losses : np.ndarray
        Simulated aggregate annual losses.
    historical_losses : np.ndarray
        Observed historical annual losses (from data/processed/).
    ax : matplotlib.axes.Axes, optional
        Axes to plot on; a new figure/axes is created if omitted.
    """
    if ax is None:
        _, ax = plt.subplots()
    # TODO: plot exceedance probability curves for both series on log-y axis
    raise NotImplementedError("Tail comparison pending simulated and historical loss arrays")

## Save results

In [ ]:
def save_simulation_results(annual_losses: np.ndarray, metrics: dict,
                             filename: str = "monte_carlo_results.csv") -> None:
    """Persist simulated losses and summary metrics under simulation/.

    Parameters
    ----------
    annual_losses : np.ndarray
        Simulated aggregate annual losses, one row per scenario.
    metrics : dict
        Output of compute_risk_metrics, saved alongside as a sidecar JSON.
    filename : str
        Output CSV filename, written under SIMULATION_DIR.
    """
    SIMULATION_DIR.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"scenario": np.arange(len(annual_losses)), "annual_loss_eur": annual_losses}).to_csv(
        SIMULATION_DIR / filename, index=False
    )
    with open(SIMULATION_DIR / (Path(filename).stem + "_metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

## Run simulation

# TODO: once 02_distribution_fitting.ipynb produces models/*.json, run this pipeline.

In [ ]:
# frequency_params = load_model_params("poisson_frequency.json")
# severity_params = load_model_params("lognormal_severity.json")

# annual_losses = run_monte_carlo(frequency_params, severity_params, n_scenarios=N_SCENARIOS)
# metrics = compute_risk_metrics(annual_losses)

# plot_loss_distribution(annual_losses, metrics)
# plot_qq(annual_losses)
# plot_tail_comparison(annual_losses, historical_losses=...)

# save_simulation_results(annual_losses, metrics)

## Convergence check

Per PRD success criteria: standard error on VaR(95%) should be < 2% with
10,000 runs. Document the check here once results are available (e.g. by
bootstrapping VaR(95%) across sub-samples of the simulation output).